# Names: Nicholas Short, Melissa Short, Elan Wilkinson

# Project Group: General Aeronautics

# Class: AAI 540

# Project: AAI 540 Final Project

# Timeline: 1/13/25 - 2/24/25

# Necessary Libraries

In [1]:
%%time

from datetime import datetime, timedelta, timezone
import json
import os
import re
import boto3
from time import sleep
from threading import Thread

import pandas as pd

from sagemaker import get_execution_role, session, Session, image_uris
from sagemaker.s3 import S3Downloader, S3Uploader
from sagemaker.processing import ProcessingJob
from sagemaker.serializers import CSVSerializer

from sagemaker.model import Model
from sagemaker.model_monitor import DataCaptureConfig

session = Session()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
CPU times: user 1.47 s, sys: 257 ms, total: 1.73 s
Wall time: 2.99 s


# AWS region and  IAM Role

In [2]:
# Get Execution role
role = get_execution_role()
print("RoleArn:", role)

region = session.boto_region_name
print("Region:", region)

RoleArn: arn:aws:iam::747058696454:role/LabRole
Region: us-east-1


# S3 bucket and prefixes

In [7]:
# Setup S3 bucket
# You can use a different bucket, but make sure the role you chose for this notebook
# has the s3:PutObject permissions. This is the bucket into which the data is captured
bucket = session.default_bucket()
print("Demo Bucket:", bucket)
prefix = "sagemaker/FinalProject-ModelQualityMonitor-20250217"

# S3 prefixes
data_capture_prefix = f"{prefix}/datacapture"
s3_capture_upload_path = f"s3://{bucket}/{data_capture_prefix}"

ground_truth_upload_path = (
    f"s3://{bucket}/{prefix}/ground_truth_data/{datetime.now():%Y-%m-%d-%H-%M-%S}"
)

reports_prefix = f"{prefix}/reports"
s3_report_path = f"s3://{bucket}/{reports_prefix}"

# Get the model monitor image
monitor_image_uri = image_uris.retrieve(framework="model-monitor", region=region)

print("Image URI:", monitor_image_uri)
print(f"Capture path: {s3_capture_upload_path}")
print(f"Ground truth path: {ground_truth_upload_path}")
print(f"Report path: {s3_report_path}")

Demo Bucket: sagemaker-us-east-1-747058696454
Image URI: 156813124566.dkr.ecr.us-east-1.amazonaws.com/sagemaker-model-monitor-analyzer
Capture path: s3://sagemaker-us-east-1-747058696454/sagemaker/FinalProject-ModelQualityMonitor-20250217/datacapture
Ground truth path: s3://sagemaker-us-east-1-747058696454/sagemaker/FinalProject-ModelQualityMonitor-20250217/ground_truth_data/2025-02-18-02-39-02
Report path: s3://sagemaker-us-east-1-747058696454/sagemaker/FinalProject-ModelQualityMonitor-20250217/reports


In [8]:
S3Uploader.upload("test_data_2/upload-test-file.txt", f"s3://{bucket}/test_upload")
print("Success! You are all set to proceed.")

Success! You are all set to proceed.


# Upload model to S3

In [39]:
import tensorflow as tf

model_path = "aai-540-labs/AAI 540 Final Project/models/lstm_model.tar.gz"  # Update with actual path
if tf.saved_model.contains_saved_model(model_path):
    print("✅ Model is in SavedModel format")
else:
    print("❌ Model is NOT in SavedModel format")

❌ Model is NOT in SavedModel format


In [9]:
# Upload the pretrained model to S3
s3_key = f"s3://{bucket}/{prefix}"
model_url_1 = S3Uploader.upload("models/rf_model.tar.gz", s3_key)
model_url_2 = S3Uploader.upload("models/lstm_model.tar.gz", s3_key)
print(model_url_1)
print(model_url_2)

s3://sagemaker-us-east-1-747058696454/sagemaker/FinalProject-ModelQualityMonitor-20250217/rf_model.tar.gz
s3://sagemaker-us-east-1-747058696454/sagemaker/FinalProject-ModelQualityMonitor-20250217/lstm_model.tar.gz


# Create Model Entities

In [27]:
model_name = f"FINAL-lstm-pred-model-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"

instance_type = 'ml.m5.large'

image_uri = image_uris.retrieve(framework = "tensorflow", version = "2.10", region = region, image_scope = 'inference', instance_type = instance_type)

model = Model(image_uri = image_uri, model_data= model_url_1, role = role, sagemaker_session = session)

# Deploy Model w/ Data Capture Enabled

In [28]:
import tarfile

# S3 details
s3_bucket = "sagemaker-us-east-1-747058696454"
s3_key = "sagemaker/FinalProject-ModelQualityMonitor-20250217/lstm_model.tar.gz"
local_model_path = "/tmp/lstm_model.tar.gz"

# Download the model
s3 = boto3.client("s3")
s3.download_file(s3_bucket, s3_key, local_model_path)

# Extract the model
with tarfile.open(local_model_path, "r:gz") as tar:
    tar.extractall("/tmp/lstm_model")  # Extract to a temporary directory
    tar.list()  # List extracted files

?rwx------ root/root          0 2025-02-16 00:11:56 LSTMmodel/ 
?rwx------ root/root          0 2025-02-16 00:11:56 LSTMmodel/assets/ 
?rw------- root/root         56 2025-02-16 00:11:56 LSTMmodel/fingerprint.pb 
?rw------- root/root      69671 2025-02-16 00:11:56 LSTMmodel/saved_model.pb 
?rwx------ root/root          0 2025-02-16 00:11:56 LSTMmodel/variables/ 
?rw------- root/root     181776 2025-02-16 00:11:56 LSTMmodel/variables/variables.data-00000-of-00001 
?rw------- root/root        762 2025-02-16 00:11:56 LSTMmodel/variables/variables.index 


In [ ]:
endpoint_name = f"Final-LSTM-model-quality-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"
print("EndpointName =", endpoint_name)

data_capture_config = DataCaptureConfig(
    enable_capture = True, sampling_percentage = 100, destination_s3_uri = s3_capture_upload_path
)

model.deploy(
    initial_instance_count = 1,
    instance_type = instance_type,
    endpoint_name = endpoint_name,
    data_capture_config = data_capture_config,
)

EndpointName = Final-LSTM-model-quality-monitor-2025-02-18-0358
-------------------------------------------*

UnexpectedStatusException: Error hosting endpoint Final-LSTM-model-quality-monitor-2025-02-18-0358: Failed. Reason: The primary container for production variant AllTraffic did not pass the ping health check. Please check CloudWatch logs for this endpoint.. Try changing the instance type or reference the troubleshooting page https://docs.aws.amazon.com/sagemaker/latest/dg/async-inference-troubleshooting.html